# 提取有效特征数值

In [ ]:
%load_ext autoreload
%autoreload 2
import os

In [ ]:
# path_home_deeptan = "/mnt/hdd2/homext/wuch/xn2p"
# path_predictions = os.path.join(path_home_deeptan, "run", "predict", "deeptan")
# path_raw_df = os.path.join(path_home_deeptan, "data", "raw_df")

In [ ]:
from litdata import StreamingDataset, StreamingDataLoader
from deeptan.utils.uni import collate_fn
import torch
import polars as pl


def pick_avail_feat_value(X_pred: pl.DataFrame, litdata_dir: str):
    """
    Pick available feature values from the true and predicted dataframes.
    """
    obs_names = X_pred["obs_names"].to_list()
    var_names = X_pred.columns[1:]

    avail_true = []
    avail_pred = []
    node_names = []
    samp_names = []
    _X_pred = X_pred.__copy__()

    # Iter over each sample
    dataset = StreamingDataset(litdata_dir)
    dataloader = StreamingDataLoader(dataset, batch_size=1, collate_fn=collate_fn)
    _count = 0
    for batch in dataloader:
        _node_names = batch["node_names"][0]
        # 从 litdata 读取真实值
        _x = batch["x"].squeeze().numpy()
        # _x 的长度应该等于 _node_names 的长度
        if len(_node_names) < 1:
            continue
        for i, _var_name in enumerate(_node_names):
            if _var_name not in var_names:
                raise ValueError(f"Variable name {_var_name} not found in the true dataframe.")
            samp_names.append(obs_names[_count])
            node_names.append(_var_name)
            avail_true.append(_x[i])
            avail_pred.append(_X_pred.row(index=_count, named=True)[_var_name])
        _count += 1

    df_avail = pl.DataFrame({"samp_names": samp_names, "node_names": node_names, "avail_true": avail_true, "avail_pred": avail_pred})
    return df_avail


def read_litdata_graph(_dir: str):
    dataset = StreamingDataset(_dir)
    dataloader = StreamingDataLoader(dataset, batch_size=1, collate_fn=collate_fn)
    _count = 0
    for batch in dataloader:
        # Check if all values in batch["x"] > 1e-6
        if torch.all(batch["x"] > 1e-6):
            # print("All values in batch['x'] are greater than 1e-6.")
            print(f".   {_count}   x.shape: {batch['x'].shape}  {len(batch['node_names'][0])}")
        else:
            # print("Some values in batch['x'] are less than or equal to 1e-6.")
            print(f"X   {_count}")
        _count += 1

In [17]:
_litdata_dir = "/mnt/hdd2/homext/wuch/xn2p/data/optimized_data/sc_multiome_minmi0.35_top2000/seed_42/trn"

read_litdata_graph(_litdata_dir)

.   0   x.shape: torch.Size([301, 1])  301
.   1   x.shape: torch.Size([148, 1])  148
.   2   x.shape: torch.Size([100, 1])  100
.   3   x.shape: torch.Size([80, 1])  80
.   4   x.shape: torch.Size([130, 1])  130
.   5   x.shape: torch.Size([105, 1])  105
.   6   x.shape: torch.Size([106, 1])  106
.   7   x.shape: torch.Size([163, 1])  163
.   8   x.shape: torch.Size([141, 1])  141
.   9   x.shape: torch.Size([289, 1])  289
.   10   x.shape: torch.Size([83, 1])  83
.   11   x.shape: torch.Size([125, 1])  125
.   12   x.shape: torch.Size([124, 1])  124
.   13   x.shape: torch.Size([100, 1])  100
.   14   x.shape: torch.Size([98, 1])  98
.   15   x.shape: torch.Size([118, 1])  118
.   16   x.shape: torch.Size([154, 1])  154
.   17   x.shape: torch.Size([135, 1])  135
.   18   x.shape: torch.Size([86, 1])  86
.   19   x.shape: torch.Size([292, 1])  292
.   20   x.shape: torch.Size([98, 1])  98
.   21   x.shape: torch.Size([108, 1])  108
.   22   x.shape: torch.Size([190, 1])  190
.   23  